# Website topic from HTML structure (language-agnostic)

Predicts one of 11 topics from the page's HTML **without looking at words** -- tag tree,
attribute usage, link syntax, metadata/schema, platform fingerprints, class/id naming and
a script-independent shape profile of the text.

Everything here comes from `structural/`; to rebuild from scratch (run from
`model_training/structural`, details in `README.md`):

```bash
python -m structural.train prep  --data ../27_08_01_en_cz.jsonl --workers 14
python -m structural.train train --data ../27_08_01_en_cz.jsonl --ablation   # feature ladder
python -m structural.train train --data ../27_08_01_en_cz.jsonl --sweep \
    --min-df 4,10,25 --C 1,4,16 --score-norm none,doc_z \
    --save models/structural_v1.joblib                                        # the bundle below
python -m structural.train explain --model models/structural_v1.joblib --labels Shopping,News
```

Evaluation uses three splits: held-out **websites**, and trained-on-Czech/tested-on-English
plus the reverse (the language-independence test).

In [1]:
import json, sys, gzip, base64
from pathlib import Path
import numpy as np

HERE = Path.cwd() if Path.cwd().name == "structural" else Path("structural")
sys.path.insert(0, str(HERE.parent))
DATA = HERE.parent / "27_08_01_en_cz.jsonl"

from structural import data as D, features as F
from structural.pipeline import StructuralWebsiteClassifier
from structural.train import domain_splits, lang_splits, label_universe, run_config

ds = D.load(DATA, groups=F.ALL_GROUPS, workers=14)
labels, counts = label_universe(ds, min_count=10)
print(len(ds), "pages", ds.stats.shape)
print("languages:", {l: ds.langs.count(l) for l in sorted(set(ds.langs))})
print("domains  :", len(set(ds.domains)))
print("labels   :", {l: counts[l] for l in labels})

[data] loading cached features 27_08_01_en_cz_1e0f1579.joblib


8291 pages (8291, 129)
languages: {'cs': 5476, 'en': 2815}
domains  : 6259
labels   : {'Adult': 127, 'Automotive': 304, 'Computers': 1556, 'Entertainment': 1211, 'Finance': 883, 'Food': 513, 'Health': 813, 'News': 1138, 'Shopping': 2039, 'Sports': 556, 'Travel': 682}


In [2]:
# Which feature families buy what.  `core` = tags/attrs/link-syntax/metadata/platforms,
# i.e. no naming at all; `text_only` is the language-dependent upper bound.
# Hyper-parameters are the ones the sweep picked (reports/sweep.md): min_df=10, C=4,
# raw scores -- per-document z-scoring turned out to hurt once empty predictions
# were fixed by force_one decoding.
runs = {}
for name, groups in [
    ("core", ("tag", "attr", "href", "meta", "plat")),
    ("core+css", ("tag", "attr", "href", "meta", "plat", "css")),
    ("all_struct", F.STRUCT_GROUPS),
    ("text_only", ("words",)),
]:
    r = run_config(ds, name, {"groups": groups, "min_df": 10, "C": 4.0,
                              "score_norm": "none"}, labels, verbose=False)
    runs[name] = r
    print(name, "done", flush=True)

print(f"\n{'config':<12}{'cols':>8}   " + "".join(f"{s:>24}" for s in ("domain", "cs2en", "en2cs")))
for name, r in runs.items():
    cells = ""
    for sp in ("domain", "cs2en", "en2cs"):
        m = r["splits"][sp]
        cells += f"  micro {m['micro_f1']:.3f} auc {m['labeled_mean_auc']:.3f}"
    print(f"{name:<12}{r['X']:>8}{cells}")

core done


core+css done


all_struct done


text_only done



config          cols                     domain                   cs2en                   en2cs
core            7033  micro 0.425 auc 0.765  micro 0.263 auc 0.625  micro 0.234 auc 0.654
core+css       27842  micro 0.467 auc 0.792  micro 0.307 auc 0.668  micro 0.234 auc 0.673
all_struct     50019  micro 0.495 auc 0.807  micro 0.323 auc 0.682  micro 0.281 auc 0.680
text_only      35452  micro 0.757 auc 0.956  micro 0.506 auc 0.875  micro 0.591 auc 0.896


In [3]:
# Per-topic behaviour of the structural model on held-out websites.
import pandas as pd
best = runs["all_struct"]["splits"]["domain"]["per_class"]
pd.DataFrame(best).T.sort_values("n", ascending=False).round(3)

,p,r,f1,n
Shopping,0.724,0.646,0.683,288.0
Entertainment,0.515,0.630,0.567,216.0
News,0.554,0.615,0.583,208.0
Computers,0.467,0.629,0.536,194.0
Health,0.333,0.237,0.277,139.0
Finance,0.270,0.387,0.319,111.0
Sports,0.356,0.564,0.437,110.0
Food,0.414,0.238,0.302,101.0
Travel,0.300,0.219,0.253,96.0
Automotive,0.329,0.418,0.368,55.0


In [4]:
# How much of the cross-lingual loss is the representation and how much is only the
# decision thresholds?  A model moved from English to Czech pages keeps its *ranking*
# (AUC), but per-label cut-offs tuned on English validation rows stop fitting because the
# whole score level shifts.  Re-tuning the cut-offs on target-language rows -- weights and
# features untouched -- separates the two effects.  Rows are split in half so the re-tuned
# thresholds are still evaluated on pages they never saw.
from sklearn.metrics import roc_auc_score
from structural.model import binarize, tune_thresholds, report

tr, va, te = lang_splits(ds)["en2cs"]
clf2 = StructuralWebsiteClassifier.fit(ds, tr, va, labels, groups=F.STRUCT_GROUPS,
                                       min_df=10, C=4.0, verbose=False)
S = clf2.score_matrix(ds, te)
Y = binarize([ds.labels[i] for i in te], labels)
pick = np.random.default_rng(0).permutation(len(te))
A, B = pick[: len(te) // 2], pick[len(te) // 2:]        # A = re-tuning rows, B = scored

def show(tag, p):
    m = report(Y[B], p, labels)
    print(f"  {tag:<36} micro {m['micro_f1']:.3f}  macro {m['macro_f1']:.3f}  "
          f"exact {m['exact_match']:.3f}")

for k in (1, 2):                                          # threshold-free decoding
    p = np.zeros_like(Y[B])
    p[np.arange(len(B))[:, None], np.argsort(-S[B], 1)[:, :k]] = 1
    show(f"top-{k} (no thresholds)", p)

def decoded(thr):
    p = (S[B] >= np.asarray(thr)).astype(np.int8)
    empty = p.sum(1) == 0
    p[empty, S[B][empty].argmax(1)] = 1                   # same force_one rule as predict()
    return p

show("thresholds tuned on English rows", decoded(clf2.thresholds))
show("thresholds re-tuned on Czech rows", decoded(tune_thresholds(S[A], Y[A])))
print(f"\n  mean AUC of the scored half (threshold-free): "
      f"{roc_auc_score(Y[B], S[B], average=None).mean():.3f}")

  top-1 (no thresholds)                micro 0.356  macro 0.274  exact 0.317
  top-2 (no thresholds)                micro 0.348  macro 0.298  exact 0.016
  thresholds tuned on English rows     micro 0.276  macro 0.284  exact 0.067


  thresholds re-tuned on Czech rows    micro 0.334  macro 0.319  exact 0.115

  mean AUC of the scored half (threshold-free): 0.674


In [5]:
clf = StructuralWebsiteClassifier.load(HERE / "models" / "structural_v1.joblib")
print("labels:", clf.labels)
print("groups:", clf.groups)
print("vocab :", len(clf.vectorizer.vocab), "tokens +", len(clf.vectorizer.num_names), "statistics")
print(json.dumps({k: v for k, v in clf.metrics.items() if k != "cfg"}, indent=1)[:900])

labels: ['Adult', 'Automotive', 'Computers', 'Entertainment', 'Finance', 'Food', 'Health', 'News', 'Shopping', 'Sports', 'Travel']
groups: ['tag', 'attr', 'href', 'meta', 'marker', 'plat', 'res', 'css', 'cssraw', 'shape']
vocab : 58833 tokens + 129 statistics
{
 "config": "mdf10 xdf0.75 C4.0 none b0",
 "domain": {
  "micro_f1": 0.49482596425211667,
  "macro_f1": 0.46451863689095724,
  "samples_f1": 0.4826734938381804,
  "exact_match": 0.34832379394930496,
  "hamming": 0.11975024158180332,
  "empty_pred": 0.0,
  "per_class": {
   "Adult": {
    "p": 0.917,
    "r": 0.688,
    "f1": 0.786,
    "n": 16
   },
   "Automotive": {
    "p": 0.329,
    "r": 0.418,
    "f1": 0.368,
    "n": 55
   },
   "Computers": {
    "p": 0.467,
    "r": 0.629,
    "f1": 0.536,
    "n": 194
   },
   "Entertainment": {
    "p": 0.515,
    "r": 0.63,
    "f1": 0.567,
    "n": 216
   },
   "Finance": {
    "p": 0.27,
    "r": 0.387,
    "f1": 0.319,
    "n": 111
   },
   "Food": {
    "p": 0.414,
    "r": 0.238

In [6]:
# Score a few pages straight from the dataset -- the model only ever sees HTML bytes.
want = {"cs": 2, "en": 2}
seen = []
with open(DATA, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        lang = D.detect_lang(D.decompress(rec["compressed_html"]), rec["url"])
        if want.get(lang, 0) > 0:
            want[lang] -= 1
            seen.append((lang, rec))
        if sum(want.values()) == 0:
            break

for lang, rec in seen:
    raw = D.decompress(rec["compressed_html"])
    print(f"\n{lang}  {rec['url']}\n  truth      : {rec['categories']}")
    for lab, s in clf.predict(raw, url=rec["url"])[:3]:
        print(f"  {s:6.3f}   {lab}")


en  http://jv.gilead.org.il/virginia/eighty
  truth      : ['Travel']
   0.778   Computers
   0.773   Entertainment

en  http://www.teamworksmedia.com
  truth      : ['Entertainment']
   0.856   Entertainment

cs  http://www.learnhockey.com
  truth      : ['Sports']
   0.842   Sports

cs  http://www.habsworld.net
  truth      : ['Sports']
   0.831   Entertainment


## Reading the numbers

- **Structure is worth something, but it is not free.** On held-out websites the
  word-free model reaches micro F1 0.495 / AUC 0.807, against 0.757 / 0.956 for the
  text baseline that is allowed to read the words. Roughly two thirds of the signal is
  in the words; the third that is left is real and it is the part that survives
  translation, CMS migration and a page with no text at all.
- **Most of the structure comes from naming, not from the tree.** Tags alone (0.332) do
  no better than the 129 raw page statistics (0.344). Attributes are what make it work
  (0.413), and CSS class names are the single biggest addition (0.473) — CMS vocabulary
  (`product`, `price`, `cart`, `article`, `date`) is English in Czech, German and Korean
  sites alike, which is why it counts as language-free.
- **Structure names page *function*, not subject.** `Shopping` is the class that
  transfers across languages (F1 0.571 Czech→English), because a product grid plus a
  price element plus a cart form *is* a layout. `Food` collapses to 0.031: cuisine is
  content. Expect this model to separate shops, forums and article templates, not
  cuisines from car brands.
- **Part of the cross-lingual loss is only calibration.** Re-tuning thresholds on target-
  language rows with the weights frozen recovers 0.06 micro F1, and plain top-1 decoding
  beats the source-language thresholds outright (0.356 vs 0.276). Deployed on a new
  language: use top-k, do not ship the tuned thresholds.
- **The `cs2en`/`en2cs` columns are a worst case, not a clean language ablation.** The
  two halves of this corpus differ in vintage and CMS generation as well as in language —
  the English half is a 2004-era archive, and the model exploits that (see the `font` and
  `td[bgcolor]` weights under `News`), which flatters `domain` and punishes the
  cross-lingual columns.
- **Upgrade path** for content signal *and* language independence: concatenate a
  multilingual sentence embedding of the visible text onto these features.
  `Vectorizer.transform` is the only code that would have to change.

## Reading the numbers

- Structure alone lands well below the text baseline in-domain, but well above nothing:
  it is learning page *function* (shop, listing, forum, form-heavy portal), which shows up
  as `Shopping` transferring best across languages.
- Topics that are defined by subject rather than layout (`Food`, `Automotive`, `Sports`)
  are where a word-free model runs out of road; no amount of tag engineering names a
  soufflé.
- The two halves of this corpus differ in vintage as well as language, so `cs2en`/`en2cs`
  are a pessimistic bound, not a clean language ablation.
- If you need the content signal *and* language independence, the upgrade path is a
  multilingual sentence embedding of the visible text concatenated with these structural
  features; nothing in this pipeline would have to change otherwise.